# Terra Lens — Pipeline A: Full Dual-Domain Training (2-stage cascade)

**Team TechTonic.** This notebook trains the *full* Terra Lens architecture end-to-end:
edge-guided RSTB spatial branch + block-DCT frequency branch (FRN) + AC-energy guided
fusion + PixelShuffle upsampler — exactly as specified in the architecture diagram —
targeting **B2/B3/B4/B8** on SEN2VENµS.

**Two-stage cascade to bridge 5 m -> <4 m:**

1. **Stage 1 (real pairs, solid ground):** train the full network at native 2x on
   real SEN2VENµS Sentinel-2 <-> VENµS pairs. No synthetic-degradation assumption.
2. **Stage 2 (Wald's protocol):** push further, ~5 m -> ~2.5 m, self-cascading the
   Stage-1 model and fine-tuning on Wald's-protocol-degraded pairs (no real
   5 m->2.5 m reference exists, so we manufacture a verifiable one exactly the way
   the evaluation framework already does for the Wald-protocol eval leg — Figure 1
   in the architecture report — reused here for *training*, not just evaluation).

Run top-to-bottom on a Kaggle GPU kernel (T4x2 or P100). Expect Stage 1 to be the
expensive part; Stage 2 is a short, cheap fine-tune on top.


In [ ]:
# ============================================================
# 0. Environment setup (Kaggle GPU kernel: Settings -> Accelerator -> GPU T4x2 / P100)
# ============================================================
import os, sys, math, glob, random, json, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# rasterio handles Sentinel-2 / VENuS GeoTIFF band reads
try:
    import rasterio
except ImportError:
    os.system("pip install -q rasterio")
    import rasterio


## Dataset setup on Kaggle — SEN2VENµS

SEN2VENµS pairs Sentinel-2 L1C reflectance (resampled to 5 m) with VENµS
reference imagery (native ~5 m, radiometrically/geometrically corrected) across
29 sites. Terra Lens targets exactly **B2 (Blue), B3 (Green), B4 (Red), B8 (NIR)**.

**Steps to get it onto this kernel:**

1. On kaggle.com, open *Datasets → New Dataset* (or search first — SEN2VENµS is
   already mirrored by several public Kaggle datasets; search `"sen2venus"`).
   If nothing usable turns up, create your own mirror:
   - Download the site archives from the official host (IEEE DataPort /
     Zenodo mirror of SEN2VENµS, `https://zenodo.org/record/...` — check the
     paper's data-availability section for the current DOI).
   - Upload the per-site `.zip`/`.nc`/`.tif` archives as a private Kaggle
     Dataset (`kaggle datasets create -p <local_dir>` via the Kaggle CLI, or
     the web uploader — Kaggle datasets support up to ~100 GB per dataset,
     which comfortably covers all 29 sites).
2. In this notebook: **Add Data → your SEN2VENµS dataset** (right-hand panel).
   It will mount read-only at `/kaggle/input/<dataset-slug>/`.
3. Point `DATA_ROOT` below at that mount. The loader below expects, per site,
   a folder containing the four bands for both sensors:
   `S2_<site>_<date>_B{02,03,04,08}.tif` (5 m Sentinel-2) and
   `VENUS_<site>_<date>_B{...}.tif` (VENµS reference) — adjust `GLOB_PATTERNS`
   to match whatever the mirror you find actually ships; SEN2VENµS is
   distributed as per-patch `.nc`/`.pt`/`.npy` bundles depending on the
   mirror, so this cell is written defensively (tries GeoTIFF first, falls
   back to `.npy`/`.pt` stacks of shape `[4,H,W]` in band order B2,B3,B4,B8).
4. **Split by site, not by patch.** Patches from the same site/date are
   spatially correlated (same field boundaries, same cloud/atmosphere), so a
   patch-level split leaks information across train/val/test and inflates
   your reported PSNR/SSIM. The code below hashes site names into
   train/val/test buckets deterministically.


In [ ]:
# ============================================================
# 1. Dataset: SEN2VENµS, split by SITE (not by patch) to avoid leakage
# ============================================================
DATA_ROOT = "/kaggle/input/sen2venus"   # <-- point this at your mounted dataset
BANDS = ["B02", "B03", "B04", "B08"]     # Blue, Green, Red, NIR  (Terra Lens target)

def discover_sites(root):
    """Each subfolder under root is treated as one site/date acquisition."""
    sites = sorted([d for d in glob.glob(os.path.join(root, "*")) if os.path.isdir(d)])
    if not sites:
        # flat layout fallback: group files by the <site> token in the filename
        files = glob.glob(os.path.join(root, "*"))
        sites = sorted(set("_".join(os.path.basename(f).split("_")[:2]) for f in files))
    return sites

def site_level_split(sites, val_frac=0.15, test_frac=0.15, seed=SEED):
    rng = random.Random(seed)
    sites = sites[:]
    rng.shuffle(sites)
    n = len(sites)
    n_test = max(1, int(n * test_frac))
    n_val = max(1, int(n * val_frac))
    test_sites = sites[:n_test]
    val_sites = sites[n_test:n_test + n_val]
    train_sites = sites[n_test + n_val:]
    return train_sites, val_sites, test_sites

def _load_band_stack(path_no_ext):
    """Try GeoTIFF (per-band files), then .npy, then .pt — whatever the mirror ships."""
    tif_paths = [f"{path_no_ext}_{b}.tif" for b in BANDS]
    if all(os.path.exists(p) for p in tif_paths):
        arrs = []
        for p in tif_paths:
            with rasterio.open(p) as src:
                arrs.append(src.read(1).astype(np.float32))
        return np.stack(arrs, axis=0)
    if os.path.exists(path_no_ext + ".npy"):
        return np.load(path_no_ext + ".npy").astype(np.float32)
    if os.path.exists(path_no_ext + ".pt"):
        return torch.load(path_no_ext + ".pt").numpy().astype(np.float32)
    raise FileNotFoundError(f"No band stack found for {path_no_ext}")

class Sen2VenusPairs(Dataset):
    """
    Returns (LR, HR) 4-band (B2,B3,B4,B8) reflectance pairs, normalized to [0,1]
    via a fixed reflectance scale (Sentinel-2 L1C digital numbers /10000).
    LR = Sentinel-2 side (5 m), HR = VENµS side (native, ~2.3x finer).
    """
    def __init__(self, site_dirs, patch_size=96, scale=2, augment=True, refl_scale=10000.0):
        self.pairs = []
        for sd in site_dirs:
            lr_candidates = sorted(glob.glob(os.path.join(sd, "*S2*")))
            hr_candidates = sorted(glob.glob(os.path.join(sd, "*VENUS*")))
            # naive pairing by index; adjust to real filename convention once known
            for lr, hr in zip(lr_candidates, hr_candidates):
                self.pairs.append((lr, hr))
        self.patch_size = patch_size
        self.scale = scale
        self.augment = augment
        self.refl_scale = refl_scale

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        lr_path, hr_path = self.pairs[idx]
        lr = _load_band_stack(os.path.splitext(lr_path)[0]) / self.refl_scale
        hr = _load_band_stack(os.path.splitext(hr_path)[0]) / self.refl_scale

        ps = self.patch_size
        h, w = lr.shape[-2:]
        y = random.randint(0, max(0, h - ps))
        x = random.randint(0, max(0, w - ps))
        lr_p = lr[:, y:y + ps, x:x + ps]
        hr_p = hr[:, y * self.scale:(y + ps) * self.scale, x * self.scale:(x + ps) * self.scale]

        if self.augment:
            if random.random() < 0.5:
                lr_p = np.ascontiguousarray(lr_p[:, :, ::-1]); hr_p = np.ascontiguousarray(hr_p[:, :, ::-1])
            if random.random() < 0.5:
                lr_p = np.ascontiguousarray(lr_p[:, ::-1, :]); hr_p = np.ascontiguousarray(hr_p[:, ::-1, :])
            k = random.randint(0, 3)
            lr_p = np.ascontiguousarray(np.rot90(lr_p, k, axes=(1, 2)))
            hr_p = np.ascontiguousarray(np.rot90(hr_p, k, axes=(1, 2)))

        return torch.from_numpy(lr_p.copy()).float(), torch.from_numpy(hr_p.copy()).float()

sites_all = discover_sites(DATA_ROOT)
print(f"Discovered {len(sites_all)} candidate sites under {DATA_ROOT}")
train_sites, val_sites, test_sites = site_level_split(sites_all)
print(f"train sites: {len(train_sites)} | val sites: {len(val_sites)} | test sites: {len(test_sites)}")


## 2. Degradation model (Wald's protocol: B, S, R operators)

Used to build the Stage-2 supervised pairs and for both evaluation legs.

In [ ]:
# ============================================================
# 2. Degradation model — Wald's protocol operators B (blur), S (subsample), R (sensor response)
#    Used to synthesize LR/reference pairs when no real 5m -> ~2.5m ground truth exists.
# ============================================================
def gaussian_kernel2d(kernel_size=9, sigma=1.6, device="cpu"):
    ax = torch.arange(kernel_size, device=device) - (kernel_size - 1) / 2.0
    xx, yy = torch.meshgrid(ax, ax, indexing="ij")
    k = torch.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return (k / k.sum()).float()

class WaldDegradation(nn.Module):
    """
    B: sensor-MTF-matched Gaussian blur (per-band, depthwise conv, fixed).
    S: nearest/area subsampling by `scale`.
    R: sensor spectral-response perturbation (mild per-band gain jitter),
       applied only during training for robustness, off at eval time.
    """
    def __init__(self, n_bands=4, scale=2, kernel_size=9, sigma=1.6, jitter=0.02):
        super().__init__()
        k = gaussian_kernel2d(kernel_size, sigma)
        self.register_buffer("kernel", k.view(1, 1, kernel_size, kernel_size).repeat(n_bands, 1, 1, 1))
        self.scale = scale
        self.n_bands = n_bands
        self.jitter = jitter
        self.pad = kernel_size // 2

    def forward(self, x, training_jitter=False):
        # B: blur
        xb = F.conv2d(F.pad(x, [self.pad] * 4, mode="reflect"), self.kernel, groups=self.n_bands)
        # S: subsample
        xs = xb[:, :, ::self.scale, ::self.scale]
        # R: sensor-response gain jitter (train-time only, simulates cross-sensor mismatch)
        if training_jitter:
            gain = 1.0 + (torch.rand(1, self.n_bands, 1, 1, device=x.device) - 0.5) * 2 * self.jitter
            xs = xs * gain
        return xs

def bicubic_upsample(x, scale):
    return F.interpolate(x, scale_factor=scale, mode="bicubic", align_corners=False)


## 3. Terra Lens architecture — spatial branch, frequency branch (FRN), guided fusion, PixelShuffle upsampler

In [ ]:
# ============================================================
# 3. Terra Lens architecture
#    Spatial branch: Sobel edge map -> edge-guided RSTB x6
#    Frequency branch: block-DCT -> FRN (conv-relu-conv) -> inverse block-DCT
#    Fusion: AC-energy-map guided sigma-gated fusion
#    Upsampler: PixelShuffle x scale, + global bicubic residual
# ============================================================

# ---- 3a. Sobel edge map --------------------------------------------------
class SobelEdge(nn.Module):
    def __init__(self, n_bands):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
        ky = kx.t()
        self.register_buffer("kx", kx.view(1, 1, 3, 3).repeat(n_bands, 1, 1, 1))
        self.register_buffer("ky", ky.view(1, 1, 3, 3).repeat(n_bands, 1, 1, 1))
        self.n_bands = n_bands

    def forward(self, x):
        gx = F.conv2d(F.pad(x, [1, 1, 1, 1], mode="reflect"), self.kx, groups=self.n_bands)
        gy = F.conv2d(F.pad(x, [1, 1, 1, 1], mode="reflect"), self.ky, groups=self.n_bands)
        e = torch.sqrt(gx ** 2 + gy ** 2 + 1e-6)          # H x W edge magnitude per band
        e = e.mean(dim=1, keepdim=True)                    # collapse to single edge map
        return torch.sigmoid(e)                            # sigma(.) gate in [0,1]

# ---- 3b. Edge-guided window attention ------------------------------------
def window_partition(x, win):
    B, C, H, W = x.shape
    x = x.view(B, C, H // win, win, W // win, win)
    x = x.permute(0, 2, 4, 3, 5, 1).contiguous().view(-1, win * win, C)
    return x  # (B*nWin, win*win, C)

def window_reverse(windows, win, H, W, B):
    C = windows.shape[-1]
    x = windows.view(B, H // win, W // win, win, win, C)
    x = x.permute(0, 5, 1, 3, 2, 4).contiguous().view(B, C, H, W)
    return x

class EdgeGuidedWindowAttention(nn.Module):
    def __init__(self, dim, window_size=8, num_heads=6):
        super().__init__()
        self.dim = dim
        self.win = window_size
        self.heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)
        self.rel_pos = nn.Parameter(torch.zeros((2 * window_size - 1) ** 2, num_heads))
        coords = torch.stack(torch.meshgrid(torch.arange(window_size), torch.arange(window_size), indexing="ij"))
        coords = coords.flatten(1)
        rel = coords[:, :, None] - coords[:, None, :]
        rel = rel.permute(1, 2, 0).contiguous()
        rel[:, :, 0] += window_size - 1
        rel[:, :, 1] += window_size - 1
        rel[:, :, 0] *= 2 * window_size - 1
        self.register_buffer("rel_index", rel.sum(-1))
        nn.init.trunc_normal_(self.rel_pos, std=0.02)

    def forward(self, x_tokens, edge_gate_tokens):
        # x_tokens: (nW, N, C); edge_gate_tokens: (nW, N, 1) in [0,1]
        nW, N, C = x_tokens.shape
        qkv = self.qkv(x_tokens).reshape(nW, N, 3, self.heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        bias = self.rel_pos[self.rel_index.view(-1)].view(N, N, self.heads).permute(2, 0, 1)
        attn = attn + bias.unsqueeze(0)
        attn = attn.softmax(dim=-1)
        # edge gate modulates the attention map before aggregating V (per diagram: attn (.) edge -> x V)
        gate = edge_gate_tokens.squeeze(-1).unsqueeze(1).unsqueeze(2)  # (nW,1,1,N)
        attn = attn * gate
        attn = attn / (attn.sum(-1, keepdim=True) + 1e-6)
        out = (attn @ v).transpose(1, 2).reshape(nW, N, C)
        return self.proj(out)

class EdgeGuidedTransformerBlock(nn.Module):
    def __init__(self, dim, window_size=8, num_heads=6, mlp_ratio=2.0):
        super().__init__()
        self.win = window_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = EdgeGuidedWindowAttention(dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

    def forward(self, x, edge_map):
        # x: (B,C,H,W), edge_map: (B,1,H,W) in [0,1]
        B, C, H, W = x.shape
        win = self.win
        pad_h = (win - H % win) % win
        pad_w = (win - W % win) % win
        xp = F.pad(x, [0, pad_w, 0, pad_h], mode="reflect")
        ep = F.pad(edge_map, [0, pad_w, 0, pad_h], mode="reflect")
        Hp, Wp = xp.shape[-2:]

        tok = window_partition(xp, win)                       # (B*nW, N, C)
        etok = window_partition(ep, win)                      # (B*nW, N, 1)

        shortcut = tok
        tok = self.norm1(tok)
        tok = self.attn(tok, etok)
        tok = shortcut + tok
        tok = tok + self.mlp(self.norm2(tok))

        out = window_reverse(tok, win, Hp, Wp, B)
        return out[:, :, :H, :W]

class RSTB(nn.Module):
    """Residual Swin-Transformer-style Block: a few edge-guided attention blocks + conv, with residual."""
    def __init__(self, dim, depth=2, window_size=8, num_heads=6):
        super().__init__()
        self.blocks = nn.ModuleList([EdgeGuidedTransformerBlock(dim, window_size, num_heads) for _ in range(depth)])
        self.conv = nn.Conv2d(dim, dim, 3, 1, 1)

    def forward(self, x, edge_map):
        res = x
        for blk in self.blocks:
            x = blk(x, edge_map)
        x = self.conv(x)
        return x + res

# ---- 3c. Frequency branch: block-DCT -> FRN -> inverse block-DCT --------
def _dct_basis(n=8):
    basis = np.zeros((n, n), dtype=np.float32)
    for k in range(n):
        for i in range(n):
            ck = math.sqrt(1 / n) if k == 0 else math.sqrt(2 / n)
            basis[k, i] = ck * math.cos(math.pi * (2 * i + 1) * k / (2 * n))
    return torch.from_numpy(basis)  # (n,n), orthonormal DCT-II matrix

class BlockDCT(nn.Module):
    """8x8 block DCT / inverse-DCT applied per-channel via unfold, fully differentiable."""
    def __init__(self, block=8):
        super().__init__()
        self.block = block
        D = _dct_basis(block)
        self.register_buffer("D", D)
        self.register_buffer("Dt", D.t())

    def forward(self, x, inverse=False):
        B, C, H, W = x.shape
        b = self.block
        pad_h = (b - H % b) % b
        pad_w = (b - W % b) % b
        xp = F.pad(x, [0, pad_w, 0, pad_h], mode="reflect")
        Hp, Wp = xp.shape[-2:]
        patches = xp.unfold(2, b, b).unfold(3, b, b)            # B,C,Hp/b,Wp/b,b,b
        nH, nW = patches.shape[2], patches.shape[3]
        patches = patches.reshape(-1, b, b)
        D, Dt = (self.Dt, self.D) if inverse else (self.D, self.Dt)
        out = D @ patches @ Dt
        out = out.reshape(B, C, nH, nW, b, b).permute(0, 1, 2, 4, 3, 5).reshape(B, C, Hp, Wp)
        return out[:, :, :H, :W]

class FRN(nn.Module):
    """Frequency-domain reconstruction net: conv-relu-conv on DCT coefficients, then inverse DCT."""
    def __init__(self, n_bands=4, feat=32, block=8):
        super().__init__()
        self.dct = BlockDCT(block)
        self.body = nn.Sequential(
            nn.Conv2d(n_bands, feat, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(feat, n_bands, 3, 1, 1),
        )

    def forward(self, x):
        D = self.dct(x, inverse=False)
        D_hat = self.body(D)
        x_freq = self.dct(D_hat, inverse=True)
        ac_energy = (D_hat ** 2).mean(dim=1, keepdim=True)   # AC-energy map E_ac (used by the fusion gate)
        return x_freq, ac_energy

# ---- 3d. Guided fusion (AC-energy-map, sigma-gated) ----------------------
class GuidedFusion(nn.Module):
    def __init__(self, n_bands=4, feat=16):
        super().__init__()
        self.gate = nn.Sequential(nn.Conv2d(1, feat, 1), nn.ReLU(inplace=True), nn.Conv2d(feat, 1, 1))

    def forward(self, x_spatial, x_freq, ac_energy):
        g = torch.sigmoid(self.gate(ac_energy))               # sigma(.) gate, in [0,1]
        fused = g * x_freq + (1 - g) * x_spatial
        return fused, g

# ---- 3e. PixelShuffle upsampler ------------------------------------------
class PixelShuffleUpsampler(nn.Module):
    def __init__(self, n_bands, scale, feat=64):
        super().__init__()
        self.scale = scale
        self.expand = nn.Conv2d(n_bands, feat * scale * scale, 3, 1, 1)
        self.ps = nn.PixelShuffle(scale)
        self.out = nn.Conv2d(feat, n_bands, 3, 1, 1)

    def forward(self, x):
        return self.out(self.ps(self.expand(x)))

# ---- 3f. Full Terra Lens model -------------------------------------------
class TerraLens(nn.Module):
    def __init__(self, n_bands=4, dim=60, depth_per_rstb=2, n_rstb=6, window_size=8, num_heads=6, scale=2):
        super().__init__()
        self.n_bands = n_bands
        self.scale = scale
        self.edge = SobelEdge(n_bands)
        self.shallow = nn.Conv2d(n_bands, dim, 3, 1, 1)
        self.rstbs = nn.ModuleList([RSTB(dim, depth_per_rstb, window_size, num_heads) for _ in range(n_rstb)])
        self.spatial_out = nn.Conv2d(dim, n_bands, 3, 1, 1)
        self.frn = FRN(n_bands, feat=32)
        self.fusion = GuidedFusion(n_bands)
        self.upsampler = PixelShuffleUpsampler(n_bands, scale)

    def forward(self, lr):
        edge_map = self.edge(lr)                              # H x W edge map, sigma-gated
        feat = self.shallow(lr)
        for rstb in self.rstbs:
            feat = rstb(feat, edge_map)
        x_spatial = self.spatial_out(feat)

        x_freq, ac_energy = self.frn(lr)
        fused, gate = self.fusion(x_spatial, x_freq, ac_energy)

        sr = self.upsampler(fused)
        sr = sr + bicubic_upsample(lr, self.scale)             # global residual
        return sr, {"edge_map": edge_map, "gate": gate, "x_spatial": x_spatial, "x_freq": x_freq}


## 4. Band-informed inflation (B2/B3/B4/B8)

Only needed here if you warm-start the spatial-branch stem from an RGB-pretrained conv (optional for Pipeline A, since we're training from scratch by default — kept for parity with Notebook B and for ablations).

In [ ]:
# ============================================================
# 4. Band-informed inflation: adapt an RGB-pretrained conv stem to B2/B3/B4/B8
#    B4(Red)->R, B3(Green)->G, B2(Blue)->B, B8(NIR) initialized from the Red filter
#    (NIR edge structure correlates with red-channel structure more than G/B).
# ============================================================
@torch.no_grad()
def inflate_rgb_conv_to_b2b3b4b8(conv_rgb: nn.Conv2d) -> nn.Conv2d:
    """
    conv_rgb: nn.Conv2d with in_channels == 3, weight layout [out,3,kh,kw] assumed RGB order.
    Returns a new nn.Conv2d with in_channels == 4, band order [B2,B3,B4,B8] = [Blue,Green,Red,NIR].
    """
    assert conv_rgb.in_channels == 3
    w = conv_rgb.weight.data  # [out, 3, kh, kw] in R,G,B order
    out_c, _, kh, kw = w.shape
    new_w = torch.zeros(out_c, 4, kh, kw, dtype=w.dtype, device=w.device)
    R, G, B = w[:, 0], w[:, 1], w[:, 2]
    new_w[:, 0] = B         # B2 Blue  <- pretrained Blue filter
    new_w[:, 1] = G         # B3 Green <- pretrained Green filter
    new_w[:, 2] = R         # B4 Red   <- pretrained Red filter
    new_w[:, 3] = R.clone() # B8 NIR   <- pretrained Red filter (edge-structure proxy, not RGB average)

    new_conv = nn.Conv2d(4, out_c, (kh, kw), stride=conv_rgb.stride, padding=conv_rgb.padding,
                          bias=(conv_rgb.bias is not None))
    new_conv.weight.data.copy_(new_w)
    if conv_rgb.bias is not None:
        new_conv.bias.data.copy_(conv_rgb.bias.data)
    return new_conv


## 5. Losses and metrics

In [ ]:
# ============================================================
# 6. Loss: Charbonnier (robust L1) + frequency-domain term + SAM regularizer
# ============================================================
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps

    def forward(self, x, y):
        return torch.mean(torch.sqrt((x - y) ** 2 + self.eps ** 2))

charbonnier = CharbonnierLoss()
dct_for_loss = BlockDCT(block=8).to(DEVICE)

def terra_lens_loss(sr, hr, w_freq=0.1, w_sam=0.05):
    l_pix = charbonnier(sr, hr)
    l_freq = charbonnier(dct_for_loss(sr), dct_for_loss(hr))
    l_sam = sam(sr, hr) / 180.0  # normalize to ~[0,1]
    return l_pix + w_freq * l_freq + w_sam * l_sam, {"pix": l_pix.item(), "freq": l_freq.item(), "sam": l_sam.item()}


In [ ]:
# ============================================================
# 5. Metrics — Wald protocol leg (needs simulated GT) + full-res leg (no GT needed)
# ============================================================
def psnr(sr, hr, max_val=1.0):
    mse = F.mse_loss(sr, hr)
    return 10 * torch.log10((max_val ** 2) / (mse + 1e-10))

def rmse(sr, hr):
    return torch.sqrt(F.mse_loss(sr, hr) + 1e-12)

def sam(sr, hr, eps=1e-8):
    # Spectral Angle Mapper, averaged over pixels, in degrees
    dot = (sr * hr).sum(dim=1)
    n1 = sr.norm(dim=1); n2 = hr.norm(dim=1)
    cos = (dot / (n1 * n2 + eps)).clamp(-1 + 1e-7, 1 - 1e-7)
    ang = torch.acos(cos)
    return ang.mean() * 180.0 / math.pi

def ergas(sr, hr, scale, eps=1e-8):
    # Erreur Relative Globale Adimensionnelle de Synthese
    n_bands = sr.shape[1]
    total = 0.0
    for b in range(n_bands):
        rmse_b = torch.sqrt(F.mse_loss(sr[:, b], hr[:, b]) + eps)
        mean_b = hr[:, b].mean() + eps
        total += (rmse_b / mean_b) ** 2
    return 100.0 / scale * torch.sqrt(total / n_bands)

def ndvi(x, red_idx=2, nir_idx=3, eps=1e-6):
    red, nir = x[:, red_idx], x[:, nir_idx]
    return (nir - red) / (nir + red + eps)

def ndvi_consistency(sr, hr, red_idx=2, nir_idx=3):
    """Domain-specific check: how well does SR preserve NDVI vs the reference (no hallucinated vegetation edges)."""
    return F.l1_loss(ndvi(sr, red_idx, nir_idx), ndvi(hr, red_idx, nir_idx))

@torch.no_grad()
def wald_protocol_eval(model, loader, degradation, scale, device=DEVICE):
    model.eval()
    stats = {"psnr": [], "rmse": [], "sam": [], "ergas": [], "ndvi": []}
    for lr_native, hr_native in loader:
        # Wald: degrade one scale further so the *native* LR/HR pair itself becomes verifiable ground truth
        hr = hr_native.to(device)
        lr_sim = degradation(hr, training_jitter=False).to(device)
        sr, _ = model(lr_sim)
        stats["psnr"].append(psnr(sr, hr).item())
        stats["rmse"].append(rmse(sr, hr).item())
        stats["sam"].append(sam(sr, hr).item())
        stats["ergas"].append(ergas(sr, hr, scale).item())
        stats["ndvi"].append(ndvi_consistency(sr, hr).item())
    return {k: float(np.mean(v)) for k, v in stats.items()}

@torch.no_grad()
def full_res_eval(model, loader, degradation, device=DEVICE):
    """
    No ground truth at full resolution: check consistency residuals.
    e_h: SR downsampled back should match the real high-res-side input (if available).
    e_m: SR degraded through B,S,R should match the original LR input (self-consistency, Wald invariance).
    """
    model.eval()
    eh_list, em_list = [], []
    for lr_native, hr_native in loader:
        lr_native = lr_native.to(device)
        sr, _ = model(lr_native)
        # e_m: degrade SR back down, compare to original LR
        sr_degraded = degradation(sr, training_jitter=False)
        em = F.l1_loss(sr_degraded, lr_native).item()
        em_list.append(em)
        # e_h: (only if a genuine independent high-res reference exists for this patch)
        if hr_native is not None and hr_native.numel() > 0:
            hr_native = hr_native.to(device)
            eh = F.l1_loss(F.interpolate(sr, size=hr_native.shape[-2:], mode="bicubic", align_corners=False),
                            hr_native).item()
            eh_list.append(eh)
    out = {"e_m": float(np.mean(em_list))}
    if eh_list:
        out["e_h"] = float(np.mean(eh_list))
    return out


## 6. Stage 1 — full joint training on real SEN2VENµS pairs (2x)

Both branches (spatial + frequency) and the fusion gate are trained jointly from
scratch (or from a light warm start — see the optional cell below). This is the
"co-adaptation" stage: the fusion gate only learns something useful for detail
hallucination if both branches are already producing sensible, differently-biased
outputs, which is why this stage needs real, not synthetic, pairs.


In [ ]:
# ---- Config ----
SCALE_STAGE1 = 2
BATCH_SIZE = 8
PATCH_SIZE = 96
EPOCHS_STAGE1 = 60
LR_STAGE1 = 2e-4

train_ds = Sen2VenusPairs(train_sites, patch_size=PATCH_SIZE, scale=SCALE_STAGE1, augment=True)
val_ds = Sen2VenusPairs(val_sites, patch_size=PATCH_SIZE, scale=SCALE_STAGE1, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = TerraLens(n_bands=4, dim=60, depth_per_rstb=2, n_rstb=6, window_size=8,
                   num_heads=6, scale=SCALE_STAGE1).to(DEVICE)
print(f"Stage-1 model params: {sum(p.numel() for p in model.parameters())/1e6:.2f} M")

opt = torch.optim.AdamW(model.parameters(), lr=LR_STAGE1, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_STAGE1)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


In [ ]:
# ---- Stage 1 training loop ----
best_psnr = -1
CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

for epoch in range(1, EPOCHS_STAGE1 + 1):
    model.train()
    running = {"pix": 0.0, "freq": 0.0, "sam": 0.0}
    t0 = time.time()
    for lr_b, hr_b in train_loader:
        lr_b, hr_b = lr_b.to(DEVICE), hr_b.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            sr, aux = model(lr_b)
            loss, parts = terra_lens_loss(sr, hr_b)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        for k in running:
            running[k] += parts[k]
    sched.step()

    if epoch % 5 == 0 or epoch == EPOCHS_STAGE1:
        val_metrics = wald_protocol_eval(model, val_loader, WaldDegradation(4, SCALE_STAGE1).to(DEVICE), SCALE_STAGE1)
        print(f"[Stage1][epoch {epoch:03d}/{EPOCHS_STAGE1}] "
              f"loss(pix/freq/sam)={running['pix']/len(train_loader):.4f}/"
              f"{running['freq']/len(train_loader):.4f}/{running['sam']/len(train_loader):.4f} "
              f"| val PSNR={val_metrics['psnr']:.2f} SAM={val_metrics['sam']:.2f} "
              f"ERGAS={val_metrics['ergas']:.2f} | {time.time()-t0:.1f}s")
        if val_metrics["psnr"] > best_psnr:
            best_psnr = val_metrics["psnr"]
            torch.save(model.state_dict(), os.path.join(CKPT_DIR, "terra_lens_stage1_best.pt"))
            print(f"  -> saved new best (PSNR={best_psnr:.2f})")

torch.save(model.state_dict(), os.path.join(CKPT_DIR, "terra_lens_stage1_last.pt"))


## 7. Stage 2 — push 5 m -> ~2.5 m via Wald's protocol, self-cascading Stage 1

No real 5m->2.5m reference exists, so Stage 2 trains a lightweight second pass
supervised by Wald's-protocol-degraded pairs: take the *real* VENµS HR image as
"ground truth", degrade it with `WaldDegradation` to synthesize its own
LR input, then require the cascade (frozen/near-frozen Stage-1 model followed by
a small second head) to recover it. This reuses the exact two-branch evaluation
structure from the architecture report (Wald-protocol leg vs. full-res leg),
just repurposed for supervision.


In [ ]:
class Stage2Head(nn.Module):
    """Small second-pass network: refines Stage-1's upsampled output toward the next scale step."""
    def __init__(self, n_bands=4, feat=48, extra_scale=1.25):
        super().__init__()
        # extra_scale is applied via bicubic + a residual refinement conv stack
        # (net effect: Stage1 scale=2 composed with this -> ~2.5x overall, matching the report's ~2.5m target)
        self.extra_scale = extra_scale
        self.body = nn.Sequential(
            nn.Conv2d(n_bands, feat, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(feat, feat, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(feat, feat, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(feat, n_bands, 3, 1, 1),
        )

    def forward(self, x):
        up = F.interpolate(x, scale_factor=self.extra_scale, mode="bicubic", align_corners=False)
        return up + self.body(up)

class TerraLensCascade(nn.Module):
    def __init__(self, stage1: nn.Module, stage2: nn.Module):
        super().__init__()
        self.stage1 = stage1
        self.stage2 = stage2

    def forward(self, lr):
        sr1, aux = self.stage1(lr)
        sr2 = self.stage2(sr1)
        return sr2, aux

EPOCHS_STAGE2 = 20
LR_STAGE2 = 5e-5
FREEZE_STAGE1_BACKBONE = True  # only the fusion gate + upsampler stay trainable in Stage 1

model.load_state_dict(torch.load(os.path.join(CKPT_DIR, "terra_lens_stage1_best.pt")))
stage2_head = Stage2Head(n_bands=4, extra_scale=1.25).to(DEVICE)
cascade = TerraLensCascade(model, stage2_head).to(DEVICE)

if FREEZE_STAGE1_BACKBONE:
    for name, p in cascade.stage1.named_parameters():
        p.requires_grad = ("fusion" in name) or ("upsampler" in name)

wald_deg_stage2 = WaldDegradation(n_bands=4, scale=2, sigma=1.3).to(DEVICE)  # milder blur for the finer step

trainable = [p for p in cascade.parameters() if p.requires_grad]
opt2 = torch.optim.AdamW(trainable, lr=LR_STAGE2, weight_decay=1e-4)
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS_STAGE2)
print(f"Stage-2 trainable params: {sum(p.numel() for p in trainable)/1e6:.3f} M")


In [ ]:
# ---- Stage 2 training loop: Wald-protocol-degraded self-supervision ----
best_psnr2 = -1
for epoch in range(1, EPOCHS_STAGE2 + 1):
    cascade.train()
    running_loss = 0.0
    for _, hr_native in train_loader:   # reuse VENµS side as "ground truth" for this stage
        hr_ref = hr_native.to(DEVICE)                                  # pretend this is the finer target
        lr_synth = wald_deg_stage2(hr_ref, training_jitter=True)        # degrade -> synthetic LR input
        opt2.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            sr, _ = cascade(lr_synth)
            sr = F.interpolate(sr, size=hr_ref.shape[-2:], mode="bicubic", align_corners=False)
            loss, _ = terra_lens_loss(sr, hr_ref)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        opt2.step()
        running_loss += loss.item()
    sched2.step()

    if epoch % 5 == 0 or epoch == EPOCHS_STAGE2:
        val_m = wald_protocol_eval(cascade, val_loader, wald_deg_stage2, scale=2)
        print(f"[Stage2][epoch {epoch:02d}/{EPOCHS_STAGE2}] loss={running_loss/len(train_loader):.4f} "
              f"| val PSNR={val_m['psnr']:.2f} SAM={val_m['sam']:.2f} NDVI_L1={val_m['ndvi']:.4f}")
        if val_m["psnr"] > best_psnr2:
            best_psnr2 = val_m["psnr"]
            torch.save(cascade.state_dict(), os.path.join(CKPT_DIR, "terra_lens_cascade_best.pt"))
            print(f"  -> saved new best cascade (PSNR={best_psnr2:.2f})")


## 8. Final evaluation — both legs (Figure 1 of the architecture report)

- **Wald protocol eval:** PSNR / RMSE / SAM / ERGAS / NDVI-consistency against
  simulated ground truth.
- **Full-res eval:** consistency residuals `e_h`, `e_m` — no ground truth needed,
  checks that SR-then-degrade round-trips back to the real input.


In [ ]:
test_ds = Sen2VenusPairs(test_sites, patch_size=PATCH_SIZE, scale=SCALE_STAGE1, augment=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

cascade.load_state_dict(torch.load(os.path.join(CKPT_DIR, "terra_lens_cascade_best.pt")))

wald_results = wald_protocol_eval(cascade, test_loader, wald_deg_stage2, scale=2)
fullres_results = full_res_eval(cascade, test_loader, wald_deg_stage2)

print("=== Wald protocol eval (simulated GT) ===")
for k, v in wald_results.items():
    print(f"  {k.upper():8s}: {v:.4f}")

print("\n=== Full-resolution eval (no GT, consistency residuals) ===")
for k, v in fullres_results.items():
    print(f"  {k}: {v:.4f}")

with open("/kaggle/working/pipelineA_results.json", "w") as f:
    json.dump({"wald_protocol": wald_results, "full_res": fullres_results}, f, indent=2)


## 9. Quick visual sanity check


In [ ]:
import matplotlib.pyplot as plt

cascade.eval()
lr_b, hr_b = next(iter(test_loader))
lr_b, hr_b = lr_b.to(DEVICE), hr_b.to(DEVICE)
with torch.no_grad():
    sr_b, _ = cascade(lr_b)
    sr_b = F.interpolate(sr_b, size=hr_b.shape[-2:], mode="bicubic", align_corners=False)

def to_rgb(t):
    # bands are [B2,B3,B4,B8] = [Blue,Green,Red,NIR] -> display as natural-color R,G,B
    rgb = t[[2, 1, 0]].clamp(0, 1).permute(1, 2, 0).cpu().numpy()
    return rgb

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(to_rgb(F.interpolate(lr_b[0:1], size=hr_b.shape[-2:], mode="bicubic")[0])); axes[0].set_title("Bicubic LR")
axes[1].imshow(to_rgb(sr_b[0])); axes[1].set_title("Terra Lens SR (cascade)")
axes[2].imshow(to_rgb(hr_b[0])); axes[2].set_title("Reference (VENµS)")
for a in axes: a.axis("off")
plt.tight_layout(); plt.savefig("/kaggle/working/pipelineA_sample.png", dpi=150)
plt.show()
